In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import numpy as np
import pandas as pd
import os
import sys
import subprocess
import urllib.request
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score
from tqdm import tqdm

# ==========================================
# 1. 基础配置 (Configuration)
# ==========================================

class Config:
    """实验超参数配置"""
    def __init__(self):
        self.num_students = 0
        self.num_questions = 0
        self.num_concepts = 0

        # 模型参数
        self.embed_dim = 64
        self.seq_len = 100
        self.tcan_layers = 2
        self.kernel_size = 3
        # 优化: 提高 Dropout 防止过拟合
        self.dropout = 0.3

        # 训练参数
        self.batch_size = 64
        # 优化: 适当增加 Epoch 以配合学习率调度
        self.epochs = 100
        self.lr = 0.001
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

        # 数据集路径
        self.data_dir = "./data"
        self.dataset_name = "assistment-2009-2010-skill"
        self.fallback_url = "https://www.cs.wpi.edu/~gendong/data/assistments/skill_builder_data.csv"

# ==========================================
# 2. 数据处理模块 (Data Processing)
# ==========================================

class AssistDataset(Dataset):
    def __init__(self, q_ids, labels, s_ids, max_len):
        self.q_ids = q_ids
        self.labels = labels
        self.s_ids = s_ids  # 新增: 学生 ID 列表 (Scheme A)
        self.max_len = max_len

    def __len__(self):
        return len(self.q_ids)

    def __getitem__(self, idx):
        q_seq = self.q_ids[idx]
        r_seq = self.labels[idx]
        s_id = self.s_ids[idx] # 获取该序列对应的学生ID

        seq_len = len(q_seq)

        if seq_len >= self.max_len:
            q_seq = q_seq[-self.max_len:]
            r_seq = r_seq[-self.max_len:]
            mask = np.ones(self.max_len, dtype=np.float32)
        else:
            pad_len = self.max_len - seq_len
            q_seq = np.pad(q_seq, (pad_len, 0), 'constant', constant_values=0)
            r_seq = np.pad(r_seq, (pad_len, 0), 'constant', constant_values=0)
            mask = np.concatenate([np.zeros(pad_len), np.ones(seq_len)]).astype(np.float32)

        return (
            torch.tensor(q_seq, dtype=torch.long),
            torch.tensor(r_seq, dtype=torch.float32),
            torch.tensor(mask, dtype=torch.float32),
            torch.tensor(s_id, dtype=torch.long) # 返回学生ID
        )

class DataProcessor:
    def __init__(self, config):
        self.config = config
        if not os.path.exists(config.data_dir):
            os.makedirs(config.data_dir)

    def _install_edudata(self):
        """尝试自动安装 EduData"""
        print("正在尝试自动安装 EduData 库...")
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "EduData"])
            print("EduData 安装成功！")
            return True
        except Exception as e:
            print(f"自动安装失败: {e}")
            return False

    def download_data(self):
        """下载数据集 (优先 EduData，失败则用 URL)"""
        csv_file = self._find_csv(self.config.data_dir)
        if csv_file:
            print(f"发现已存在的数据文件: {csv_file}")
            return csv_file

        print(f"正在准备下载数据集: {self.config.dataset_name}")
        use_fallback = False
        try:
            from EduData import get_data
            get_data(self.config.dataset_name, self.config.data_dir)
        except ImportError:
            if self._install_edudata():
                try:
                    from EduData import get_data
                    get_data(self.config.dataset_name, self.config.data_dir)
                except Exception:
                    use_fallback = True
            else:
                use_fallback = True
        except Exception:
            use_fallback = True

        if use_fallback:
            print(f"切换至备用方案，直接下载: {self.config.fallback_url}")
            save_path = os.path.join(self.config.data_dir, "skill_builder_data.csv")
            try:
                opener = urllib.request.build_opener()
                opener.addheaders = [('User-agent', 'Mozilla/5.0')]
                urllib.request.install_opener(opener)
                urllib.request.urlretrieve(self.config.fallback_url, save_path)
                print("下载完成！")
            except Exception as e:
                raise RuntimeError(f"下载失败: {e}")

        return self._find_csv(self.config.data_dir)

    def _find_csv(self, root_dir):
        for root, dirs, files in os.walk(root_dir):
            for file in files:
                if file.endswith(".csv") and "skill" in file.lower():
                    return os.path.join(root, file)
        return None

    def process(self):
        file_path = self.download_data()
        print("读取并清洗数据...")
        try:
            df = pd.read_csv(file_path, encoding='latin-1', low_memory=False)
        except:
            df = pd.read_csv(file_path, encoding='utf-8', low_memory=False)

        # 列名标准化
        cols = df.columns
        user_col = 'user_id'
        item_col = 'problem_id'
        skill_col = next((c for c in cols if 'skill_id' in c), None)
        if not skill_col: skill_col = next((c for c in cols if 'skill_name' in c), None)
        correct_col = 'correct'
        order_col = 'order_id'

        if not skill_col:
             raise ValueError("数据集中未找到 Skill ID 列，无法进行知识点建模。")

        df.dropna(subset=[skill_col], inplace=True)
        df.drop_duplicates(subset=[user_col, item_col, order_col], inplace=True)

        # ID 映射
        def get_map(unique_vals):
            return {val: i+1 for i, val in enumerate(unique_vals)}

        user_map = get_map(df[user_col].unique())
        df['user_id_mapped'] = df[user_col].map(user_map)
        self.config.num_students = len(user_map) + 1

        prob_map = get_map(df[item_col].unique())
        df['item_id'] = df[item_col].map(prob_map)
        self.config.num_questions = len(prob_map) + 1

        skill_map = get_map(df[skill_col].unique())
        df['skill_id'] = df[skill_col].map(skill_map)
        self.config.num_concepts = len(skill_map) + 1

        # --- Optimization Scheme B: 计算题目难度 ---
        print("计算题目难度 (Difficulty)...")
        # 计算每道题的平均正确率
        item_difficulty = df.groupby('item_id')[correct_col].mean()
        # 初始化难度向量，0号为padding，难度设为0
        diff_values = np.zeros(self.config.num_questions)
        for iid, diff in item_difficulty.items():
            diff_values[iid] = diff

        print(f"统计: 学生 {self.config.num_students}, 题目 {self.config.num_questions}, 知识点 {self.config.num_concepts}")

        # --- Optimization Scheme C: 构建并归一化 Q-Matrix ---
        print("构建归一化 Q-Matrix...")
        q_k_relation = np.zeros((self.config.num_questions, self.config.num_concepts))
        tmp_df = df[['item_id', 'skill_id']].drop_duplicates()
        for _, row in tmp_df.iterrows():
            q_k_relation[int(row['item_id']), int(row['skill_id'])] = 1

        # Row Normalization: 让每个题目的知识点权重和为1，防止梯度爆炸
        row_sums = q_k_relation.sum(axis=1)
        row_sums[row_sums == 0] = 1 # 防止除以0
        q_k_relation = q_k_relation / row_sums[:, np.newaxis]

        # 生成序列
        print("生成序列...")
        df.sort_values(by=[order_col], inplace=True)
        all_q, all_r, all_s = [], [], []

        # 为了演示速度，这里默认只取前2000个学生。
        # 如果要在完整数据集上运行，请注释掉下面两行。
        users = df['user_id_mapped'].unique()[:2000]
        df_small = df[df['user_id_mapped'].isin(users)]
        # df_small = df # Uncomment this line to use full data

        for uid, group in tqdm(df_small.groupby('user_id_mapped')):
            if len(group) < 5: continue
            all_q.append(group['item_id'].values)
            all_r.append(group[correct_col].values)
            all_s.append(uid) # 记录该序列属于哪个学生

        return all_q, all_r, all_s, q_k_relation, diff_values

# ==========================================
# 3. 核心模型 (Optimized Model)
# ==========================================

class HeteroGraphEmbedding(nn.Module):
    def __init__(self, config, diff_values):
        super(HeteroGraphEmbedding, self).__init__()
        self.embed_dim = config.embed_dim
        self.question_emb = nn.Embedding(config.num_questions, config.embed_dim, padding_idx=0)
        self.concept_emb = nn.Embedding(config.num_concepts, config.embed_dim, padding_idx=0)

        # Scheme B: 难度嵌入
        # diff_values 是常数张量，不需要梯度更新
        self.register_buffer('diff_values', torch.tensor(diff_values, dtype=torch.float32))
        self.diff_proj = nn.Linear(1, config.embed_dim)

    def forward(self, question_ids, q_k_relation):
        # 1. 知识点聚合 (Scheme C: GCN-like propagation)
        # q_k_relation 已经是归一化的邻接矩阵
        k_emb_weight = self.concept_emb.weight
        q_k_agg = torch.matmul(q_k_relation, k_emb_weight)

        # 2. 难度嵌入 (Scheme B)
        # 获取当前batch题目的难度值
        d_val = self.diff_values[question_ids].unsqueeze(-1)
        d_feat = self.diff_proj(d_val)

        # 3. 融合: ID + Knowledge + Difficulty
        # 查表得到 ID Embedding
        q_id_feat = self.question_emb(question_ids)

        # q_k_agg 是所有题目的聚合表，查表取出当前batch
        q_k_feat = F.embedding(question_ids, q_k_agg, padding_idx=0)

        final_q_emb = q_id_feat + q_k_feat + d_feat
        return final_q_emb

class TemporalAttention(nn.Module):
    def __init__(self, embed_dim, dropout=0.1):
        super(TemporalAttention, self).__init__()
        self.query = nn.Linear(embed_dim, embed_dim)
        self.key = nn.Linear(embed_dim, embed_dim)
        self.value = nn.Linear(embed_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, L, D = x.size()
        Q = self.query(x)
        K = self.key(x)
        V = self.value(x)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(D)
        # 因果掩码
        mask = torch.tril(torch.ones(L, L, device=x.device))
        scores = scores.masked_fill(mask == 0, -1e9)
        attn = F.softmax(scores, dim=-1)
        return torch.matmul(self.dropout(attn), V)

class TCANBlock(nn.Module):
    def __init__(self, embed_dim, kernel_size, dilation, dropout):
        super(TCANBlock, self).__init__()
        self.ta = TemporalAttention(embed_dim, dropout)
        self.padding = (kernel_size - 1) * dilation
        self.conv = nn.Conv1d(embed_dim, embed_dim, kernel_size, dilation=dilation, padding=self.padding)
        self.norm = nn.LayerNorm(embed_dim)
        self.dropout = nn.Dropout(dropout)
        self.relu = nn.ReLU()

    def forward(self, x):
        residual = x
        z = self.ta(x)
        z_perm = z.permute(0, 2, 1)
        conv_out = self.conv(z_perm)
        # 裁剪掉 padding 带来的多余长度
        conv_out = conv_out[:, :, :z.size(1)].permute(0, 2, 1)
        out = self.norm(conv_out + residual)
        return self.dropout(self.relu(out))

class HIG_TCAN_CD_Optimized(nn.Module):
    def __init__(self, config, q_k_relation, diff_values):
        super(HIG_TCAN_CD_Optimized, self).__init__()
        self.config = config
        self.register_buffer('q_k_relation', torch.tensor(q_k_relation, dtype=torch.float32))

        # 1. 图嵌入 (含难度)
        self.graph_emb = HeteroGraphEmbedding(config, diff_values)

        # 2. Scheme A: 学生静态嵌入
        self.student_emb = nn.Embedding(config.num_students, config.embed_dim, padding_idx=0)

        self.input_proj = nn.Linear(config.embed_dim + 1, config.embed_dim)
        self.tcan_layers = nn.ModuleList([
            TCANBlock(config.embed_dim, config.kernel_size, 2**i, config.dropout)
            for i in range(config.tcan_layers)
        ])

        # 3. 预测层: 输入维度变为 embed_dim * 3
        # (Dynamic State h_t) + (Next Question q_{t+1}) + (Student Static s)
        self.pred_mlp = nn.Sequential(
            nn.Linear(config.embed_dim * 3, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, q_ids, answers, student_ids):
        # q_ids: [B, L]
        # answers: [B, L]
        # student_ids: [B]

        q_emb = self.graph_emb(q_ids, self.q_k_relation)
        x = torch.cat([q_emb, answers.unsqueeze(-1)], dim=-1)
        x = self.input_proj(x)

        h = x
        for layer in self.tcan_layers:
            h = layer(h)

        # 获取学生静态特征并扩展到序列长度
        # s_static: [B, dim] -> [B, 1, dim] -> [B, L, dim]
        s_static = self.student_emb(student_ids).unsqueeze(1).expand(-1, q_ids.size(1), -1)

        return h, q_emb, s_static

# ==========================================
# 4. 实验流程 (Execution)
# ==========================================

def evaluate(model, loader, config):
    model.eval()
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for q, r, mask, s in loader:
            q, r, mask, s = q.to(config.device), r.to(config.device), mask.to(config.device), s.to(config.device)
            h_seq, q_seq_emb, s_static = model(q, r, s)

            h_pred = h_seq[:, :-1, :]
            q_next = q_seq_emb[:, 1:, :]
            s_static_seq = s_static[:, 1:, :]

            feat = torch.cat([h_pred, q_next, s_static_seq], dim=-1)
            preds = model.pred_mlp(feat).squeeze(-1)

            # 收集有效预测
            valid_mask = mask[:, 1:].bool()

            # 如果整个batch没有有效数据则跳过
            if valid_mask.sum() == 0:
                continue

            flat_preds = preds[valid_mask].cpu().numpy()
            flat_targets = r[:, 1:][valid_mask].cpu().numpy()

            all_preds.append(flat_preds)
            all_targets.append(flat_targets)

    if len(all_preds) == 0:
        print("没有有效的测试数据。")
        return

    # 优化: Global Metrics Calculation (全局计算指标)
    all_preds = np.concatenate(all_preds)
    all_targets = np.concatenate(all_targets)

    acc = accuracy_score(all_targets, (all_preds >= 0.5).astype(int))
    try:
        auc = roc_auc_score(all_targets, all_preds)
    except ValueError:
        auc = 0.0

    print(f"Test ACC: {acc:.4f} | Test AUC: {auc:.4f}")

def run_experiment():
    print(">>> 阶段 1: 数据准备 (Optimized Version)")
    config = Config()
    processor = DataProcessor(config)

    try:
        # 获取数据，包括 diff_values
        q_seqs, r_seqs, s_seqs, q_k_rel, diff_values = processor.process()
    except Exception as e:
        print(f"数据处理错误: {e}")
        return

    # 划分数据集 (增加 s_seqs)
    if len(q_seqs) == 0:
        print("错误：数据集为空。")
        return

    train_q, test_q, train_r, test_r, train_s, test_s = train_test_split(
        q_seqs, r_seqs, s_seqs, test_size=0.2, random_state=42
    )

    train_dataset = AssistDataset(train_q, train_r, train_s, config.seq_len)
    test_dataset = AssistDataset(test_q, test_r, test_s, config.seq_len)

    train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=config.batch_size, shuffle=False)

    print(f"\n>>> 阶段 2: 模型初始化 (Device: {config.device})")
    model = HIG_TCAN_CD_Optimized(config, q_k_rel, diff_values).to(config.device)
    optimizer = torch.optim.Adam(model.parameters(), lr=config.lr)

    # 优化: 学习率调度器 (Cosine Annealing)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config.epochs)

    print("\n>>> 阶段 3: 开始训练")
    for epoch in range(config.epochs):
        model.train()
        total_loss = 0
        steps = 0

        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{config.epochs}")
        for q, r, mask, s in pbar:
            q, r, mask, s = q.to(config.device), r.to(config.device), mask.to(config.device), s.to(config.device)

            optimizer.zero_grad()
            # 传入 student_ids (s)
            h_seq, q_seq_emb, s_static = model(q, r, s)

            h_pred = h_seq[:, :-1, :]
            q_next = q_seq_emb[:, 1:, :]
            s_static_seq = s_static[:, 1:, :] # 对齐序列

            r_target = r[:, 1:]
            mask_target = mask[:, 1:]

            # 融合三个特征
            feat = torch.cat([h_pred, q_next, s_static_seq], dim=-1)
            preds = model.pred_mlp(feat).squeeze(-1)

            loss = F.binary_cross_entropy(preds, r_target, reduction='none')
            masked_loss = (loss * mask_target).sum() / (mask_target.sum() + 1e-8)

            masked_loss.backward()
            optimizer.step()

            total_loss += masked_loss.item()
            steps += 1
            pbar.set_postfix({'Loss': f"{masked_loss.item():.4f}"})

        scheduler.step()
        print(f"Epoch {epoch+1} Avg Loss: {total_loss/steps:.4f}")
        evaluate(model, test_loader, config)

if __name__ == "__main__":
    run_experiment()

>>> 阶段 1: 数据准备 (Optimized Version)
发现已存在的数据文件: ./data/2009_skill_builder_data_corrected/skill_builder_data_corrected.csv
读取并清洗数据...
计算题目难度 (Difficulty)...
统计: 学生 4164, 题目 17752, 知识点 113
构建归一化 Q-Matrix...
生成序列...


100%|██████████| 2000/2000 [00:00<00:00, 11937.82it/s]



>>> 阶段 2: 模型初始化 (Device: cpu)

>>> 阶段 3: 开始训练


Epoch 1/100: 100%|██████████| 23/23 [00:04<00:00,  4.86it/s, Loss=0.6412]


Epoch 1 Avg Loss: 0.6480
Test ACC: 0.6444 | Test AUC: 0.6552


...

Epoch 44/100: 100%|██████████| 23/23 [00:05<00:00,  4.15it/s, Loss=0.3583]


Epoch 44 Avg Loss: 0.3526
Test ACC: 0.6819 | Test AUC: 0.7135


Epoch 45/100: 100%|██████████| 23/23 [00:04<00:00,  4.98it/s, Loss=0.3488]


Epoch 45 Avg Loss: 0.3496
Test ACC: 0.6795 | Test AUC: 0.7129


Epoch 46/100:  61%|██████    | 14/23 [00:02<00:01,  4.89it/s, Loss=0.3451]